In [1]:
import pandas as pd
import numpy as np

In [4]:
data = pd.read_csv(r"X:\PROJETS\PYTHON\systeme_prediction_panne_electricite\backend\data\Data_for_UCI_named.csv")

In [5]:
data

,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stab,stabf
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034,0.055347,unstable
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760,-0.005957,stable
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853,0.003471,unstable
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718,0.028871,unstable
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923,0.049860,unstable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2.930406,9.487627,2.376523,6.187797,3.343416,-0.658054,-1.449106,-1.236256,0.601709,0.779642,0.813512,0.608385,0.023892,unstable
9996,3.392299,1.274827,2.954947,6.894759,4.349512,-1.663661,-0.952437,-1.733414,0.502079,0.567242,0.285880,0.366120,-0.025803,stable
9997,2.364034,2.842030,8.776391,1.008906,4.299976,-1.380719,-0.943884,-1.975373,0.487838,0.986505,0.149286,0.145984,-0.031810,stable
9998,9.631511,3.994398,2.757071,7.821347,2.514755,-0.966330,-0.649915,-0.898510,0.365246,0.587558,0.889118,0.818391,0.037789,unstable


# FEATURE ENGINEERING

In [6]:
df = data.copy()

In [16]:

equipements_config = {
    'Transformateur 630 kVA': 500,
    'Transformateur 400 kVA': 320,
    'Transformateur 250 kVA': 200,
    'Transformateur 100 kVA': 80,
    'Disjoncteur Depart HTA': 1200,
    'Cellule d Arrivee MT': 1500,
    'Armoire de Distribution Publique': 150,
}

liste_equipements = list(equipements_config.keys())
np.random.seed(42)

df['type_materiel'] = np.random.choice(liste_equipements, size=len(df))
df['capacite_max_kw'] = df['type_materiel'].map(equipements_config)

# 3. Métriques physiques
df['consommation_kw'] = np.round(
    (np.abs(df['p1']) / 4.0) * df['capacite_max_kw'], 2
)
df['tension_v'] = np.round(230 - (df['tau1'] * 6.0), 1)

# 4. MODÈLE DE RISQUE PROBABILISTE ÉQUILIBRÉ
# Score de risque combiné (charge + sous-tension)
surcharge_score = df['consommation_kw'] / df['capacite_max_kw']
tension_score = (230 - df['tension_v']) / 60.0
risque = 0.55 * surcharge_score + 0.45 * tension_score

# Bruit mesuré pour éviter le sur-apprentissage et rééquilibrer la cible (54% / 46%)
bruit = np.random.normal(0, 0.025, size=len(df))
probabilite_panne = 1 / (1 + np.exp(-20 * (risque + bruit - 0.73)))

# Cible binaire basée sur la probabilité
df['panne'] = (np.random.rand(len(df)) < probabilite_panne).astype(int)

# 5. Sauvegarde dans le chemin projet
df_clean = df[
    ['type_materiel', 'capacite_max_kw', 'consommation_kw', 'tension_v', 'panne']
]
chemin_sauvegarde = r'X:\PROJETS\PYTHON\systeme_prediction_panne_electricite\backend\data\snel_equipment_data.csv'
df_clean.to_csv(chemin_sauvegarde, index=False)

print('--- DATASET PROBABILISTE ÉQUILIBRÉ ET RÉALISTE GÉNÉRÉ ---')
print(df_clean['panne'].value_counts())
print('\nPourcentages :')
print(df_clean['panne'].value_counts(normalize=True) * 100)

--- DATASET PROBABILISTE ÉQUILIBRÉ ET RÉALISTE GÉNÉRÉ ---
panne
1    5433
0    4567
Name: count, dtype: int64

Pourcentages :
panne
1    54.33
0    45.67
Name: proportion, dtype: float64
